# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayyankarar18/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane 1 — Ranking Signal Analysis.**

The starter dataset carries close to 40 candidate signals (position, CTR, engagement rate, scroll rate, AI traffic share, word count, trend) sitting next to real performance outcomes for 30,000 content items across 32 clients. Before I can build a priority score (Lane 2) or a CTR-gap ranking (Lane 4), I need to know which of these signals actually move together with visibility and engagement in the first place — otherwise any scoring system I build later would be prioritizing on signals that don't matter. Signal analysis is the natural first step: it tells me which levers are worth building on, and keeps Lane 2/4 open as a Week 4 pivot if the signal picture points that way.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

For an SEO content strategist deciding which structural and engagement signals to prioritize when planning content updates, I will build a ranked signal report from the starter dataset (and later the warehouse tables), showing which measurable signals — word count, scroll rate, engagement rate, AI traffic share, and similar — co-move with visibility and click outcomes, scored by grouped comparisons and effect size, not by a single opaque model score.

A wrong call here costs real editor hours: if I flag a signal as important that isn't actually associated with better performance, teams rework pages based on a lever that doesn't move anything. The opposite mistake — missing a signal that genuinely tracks with performance — means a real opportunity goes untouched. A plain if-statement rule isn't enough because there are dozens of candidate signals, their relationships shift by content type and by client, and the label trap (`trend_direction`/`trend_pct` are derived, not independent) means the tangle has to be worked through carefully rather than eyeballed.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Across 30,000 rows and 32 clients, 96.0% carry a usable `avg_position` (the rest are the "no data" zeros, excluded per the dataset's own gotchas). Within that usable set, `engagement_rate` barely correlates with `avg_position` (r = -0.027) — on its own, engagement isn't a strong stand-in for rank. But `scroll_rate` varies a lot by `content_type`: comparison articles average 61.8% vs. 15.5% for keyword articles, a 4x gap. That mix — one assumed signal that's basically flat, one real signal with a clean, content-type-dependent gradient — is exactly what Lane 1 is for.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape, df["client_id"].nunique(), "clients")

# avg_position == 0 means "no data", not rank zero -- exclude it
has_position = df[df["avg_position"] > 0]

# 1. How many rows actually carry a usable position signal at all
coverage = len(has_position) / len(df) * 100
print(f"Rows with a usable avg_position: {coverage:.1f}%")

# 2. Does engagement_rate move with avg_position (lower position number = better rank)
corr = has_position["engagement_rate"].corr(has_position["avg_position"])
print(f"Correlation, engagement_rate vs avg_position: {corr:.3f}")

# 3. Does scroll_rate differ meaningfully by content_type
by_type = df.groupby("content_type")["scroll_rate"].mean().sort_values(ascending=False)
print("Mean scroll_rate by content_type:")
print(by_type)

(30000, 44) 32 clients
Rows with a usable avg_position: 96.0%
Correlation, engagement_rate vs avg_position: -0.027
Mean scroll_rate by content_type:
content_type
comparison article    61.824101
feedly article        39.102797
keyword article       15.478047
Name: scroll_rate, dtype: float64


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

What this work can say: which signals are **observed** to move together with visibility and engagement outcomes, within this 90-day window, for these 32 clients; **directional** relationships (e.g., "higher scroll_rate tends to co-occur with better avg_position within this sample"); a **decision-support** ranking of signals an editor could use to decide what to look at first.

What it can never say: that any signal *causes* a ranking change; that findings generalize past this snapshot's clients, content types, or time window; anything about how Google's ranking algorithm itself works — I'm reading outcomes, not "predicting Google"; and no causal claim built on `trend_direction` or `trend_pct`, since both are derived from other columns in this same dataset rather than independent measurements.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.